In [ ]:
# AI harness: Panel chat app + MCP (stdio) tools + OpenAI-compatible models.
# Run this cell, then click the printed link: the app opens in a new tab via the Jupyter proxy.
# Call shutdown() when finished, or restart the kernel.
# To adapt this to your own OpenAI endpoint and MCP server, search for "ADAPT:" (each marks a line to replace or scratch out).
import asyncio
import html
import json
import os
import re
import sys
import threading
import time
from collections.abc import AsyncIterator
from pathlib import Path

import panel as pn
from IPython.display import HTML, display
# ADAPT: needs mcp < 2 (2.x changed the client APIs), panel >= 1.5, openai >= 1, Python 3.10+.
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import BadRequestError, OpenAI

# Re-running the cell: stop the previous app and MCP server before starting new ones.
if "server" in globals():
    server.stop()
if "bridge" in globals():
    bridge.close()


# ============================================================================ config

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> None:
    """Load KEY=VALUE pairs from a .env file without overriding variables that are already set.

    Args:
        path: Location of the .env file. A missing file is ignored.
    """
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.split(" #")[0].strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


# Settings come from .env; swap this for a secrets store if you have one.
# ADAPT: scratch out if you have no .env file; set the environment variables below some other way.
load_env(PROJECT_ROOT / ".env")

# ADAPT: your OpenAI-compatible endpoint URL and API key (env vars, hardcoded or from a secrets store).
OPENAI_BASE_URL = os.environ["OPENAI_BASE_URL"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# ADAPT: path to your MCP server script.
MCP_SERVER_SCRIPT = str(PROJECT_ROOT / "mcp_server" / "server.py")
# ADAPT: anywhere writable for the MCP server's stderr log.
MCP_LOG_PATH = PROJECT_ROOT / "logs" / "mcp_server.log"
# ADAPT: your proxy_cert path; the two lines after this one only resolve a relative path, scratch them out if it is absolute.
PROXY_CERT = os.environ.get("PROXY_CERT", "")
if PROXY_CERT and not os.path.isabs(PROXY_CERT):
    PROXY_CERT = str(PROJECT_ROOT / PROXY_CERT)
# ADAPT: replace with the environment variables your MCP server reads (proxy credentials etc.).
MCP_ENV = {
    "PROXY_USERNAME": os.environ["PROXY_USERNAME"],
    "PROXY_PASSWORD": os.environ["PROXY_PASSWORD"],
    "PROXY_CERT": PROXY_CERT,
    "PROXY_VERIFY": os.environ.get("PROXY_VERIFY", "true"),
    # ADAPT: scratch out: only this project's ticket service uses MICRO_URL.
    "MICRO_URL": os.environ.get("MICRO_URL", "https://localhost:8443"),
}

# ADAPT: a free port your Jupyter proxy can reach.
PANEL_PORT = int(os.environ.get("PANEL_PORT", 5006))
# ADAPT: set to your Jupyter hostname(s) as the browser sees them if the app page loads blank.
PANEL_WS_ORIGINS = [origin.strip() for origin in os.environ.get("PANEL_WS_ORIGINS", "*").split(",") if origin.strip()]

# ADAPT: set to your models' real context window if /v1/models doesn't report it (or use CONTEXT_OVERRIDES).
DEFAULT_CONTEXT = 4096          # used when the server doesn't report a model's window
CONTEXT_OVERRIDES: dict[str, int] = {}  # e.g. {"qwen2.5:3b": 32768} if the server reports it wrong
MAX_TOOL_STEPS = 6              # model <-> tool rounds per user message
RESERVE_OUTPUT = 768            # tokens kept free for the reply (also sent as max_tokens)
COMPACT_AT = 0.70               # auto-compact above this share of the usable window...
TARGET_AFTER = 0.45             # ...down to roughly this share
KEEP_TURNS = 2                  # most recent user turns always kept verbatim
TOOL_PREVIEW_CHARS = 300        # length of a shortened tool result

# ADAPT: rewrite for your tools: this prompt is about the ticket system.
SYSTEM_PROMPT = (
    "You are a helpful assistant with tools for a support-ticket system. "
    "Use the tools whenever the user asks about tickets, then answer briefly using the tool results. "
    "If a tool result was shortened, call recall_tool_result with its id to read the full text."
)
# ADAPT: change the fact list (ids, statuses, priorities, owners) to what matters for your tools.
SUMMARY_PROMPT = (
    "You maintain a running summary of a conversation between a user and an assistant that uses tools. "
    "Merge the NEW TRANSCRIPT into the CURRENT SUMMARY. Keep every concrete fact from tool results: "
    "ids, names, statuses, priorities, owners, numbers, dates, errors. Keep what the user asked for and decided. "
    "Drop greetings and filler. Output only the updated summary as terse bullet points, max 200 words: "
    "exactly one bullet per ticket or topic, merging new facts into the existing bullet instead of repeating it."
)
RECALL_TOOL = {"type": "function", "function": {
    "name": "recall_tool_result",
    "description": "Read the full text of an earlier tool result that was shortened to save context.",
    "parameters": {"type": "object", "properties": {
        "result_id": {"type": "string", "description": "id shown in the shortened result"},
        "offset": {"type": "integer", "default": 0},
        "length": {"type": "integer", "default": 1500},
    }, "required": ["result_id"]},
}}


def to_json(value: object) -> str:
    """Serialise a value to compact JSON, keeping non-ASCII characters readable.

    Args:
        value: Any JSON-serialisable value.

    Returns:
        The JSON string.
    """
    return json.dumps(value, ensure_ascii=False)


# ============================================================================ models

def fetch_context_sizes(openai_client: OpenAI) -> dict[str, int]:
    """List the server's models and the context window of each.

    Servers report the window under different keys (vLLM uses ``max_model_len``);
    ``CONTEXT_OVERRIDES`` wins, and ``DEFAULT_CONTEXT`` is the fallback.

    Args:
        openai_client: Client for the OpenAI-compatible server.

    Returns:
        Mapping of model id to context window size in tokens, in server order.
    """
    sizes: dict[str, int] = {}
    for model in openai_client.models.list().data:
        extra = model.model_extra or {}
        sizes[model.id] = int(CONTEXT_OVERRIDES.get(model.id) or extra.get("context_length")
                              or extra.get("max_model_len") or extra.get("context_window") or DEFAULT_CONTEXT)
    return sizes


client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
CONTEXT = fetch_context_sizes(client)
MODELS = list(CONTEXT)


# ============================================================================ MCP

class MCPBridge:
    """Keeps one MCP stdio session open on a dedicated event loop in a background thread.

    Jupyter owns one event loop and Panel serves from another thread, so the session gets its own
    loop. It is opened and closed inside a single task, which anyio's cancel scopes require.

    Attributes:
        tools: MCP tool definitions reported by the server.
    """

    def __init__(self, params: StdioServerParameters, log_path: Path) -> None:
        """Start the MCP server subprocess and wait until the session is initialised.

        Args:
            params: Command, arguments and environment for the server process.
            log_path: File that receives the server's stderr.

        Raises:
            RuntimeError: If the server does not start within 30 seconds.
        """
        self.params = params
        # ipykernel's sys.stderr has no fileno(), so the subprocess needs a real file for stderr.
        log_path.parent.mkdir(exist_ok=True)
        self.log_path = log_path
        self.log_file = open(log_path, "a")
        self.loop = asyncio.new_event_loop()
        self.thread = threading.Thread(target=self.loop.run_forever, daemon=True, name="mcp-loop")
        self.thread.start()
        self.ready = threading.Event()
        self.stop_event: asyncio.Event | None = None
        self.session: ClientSession | None = None
        self.tools: list = []
        self.main_task = asyncio.run_coroutine_threadsafe(self._run(), self.loop)
        self.ready.wait(30)
        if self.session is None:
            error = self.main_task.exception() if self.main_task.done() else None
            log_tail = "".join(open(log_path).readlines()[-5:])
            raise RuntimeError(f"MCP server failed to start ({error!r}). Last log lines:\n{log_tail}")

    async def _run(self) -> None:
        """Open the stdio session and hold it open until ``close`` is called."""
        self.stop_event = asyncio.Event()
        try:
            async with stdio_client(self.params, errlog=self.log_file) as (read_stream, write_stream):
                async with ClientSession(read_stream, write_stream) as session:
                    await session.initialize()
                    self.session = session
                    self.tools = (await session.list_tools()).tools
                    self.ready.set()
                    await self.stop_event.wait()
        finally:
            self.ready.set()

    async def acall(self, name: str, arguments: dict) -> str:
        """Call an MCP tool from any event loop (e.g. Panel's).

        Args:
            name: Tool name.
            arguments: Tool arguments.

        Returns:
            The tool's text output, or its structured content as JSON if there is no text.
        """
        future = asyncio.run_coroutine_threadsafe(self.session.call_tool(name, arguments), self.loop)
        result = await asyncio.wrap_future(future)
        text = "\n".join(block.text for block in result.content if getattr(block, "type", "") == "text")
        return text or to_json(result.structuredContent or {})

    def call(self, name: str, arguments: dict) -> str:
        """Blocking version of ``acall`` for use directly in notebook code.

        Args:
            name: Tool name.
            arguments: Tool arguments.

        Returns:
            The tool's text output.
        """
        return asyncio.run_coroutine_threadsafe(self.acall(name, arguments), self.loop).result(60)

    def openai_tools(self) -> list[dict]:
        """Convert the MCP tool definitions to the OpenAI ``tools`` format.

        Returns:
            One function definition per MCP tool.
        """
        return [{"type": "function", "function": {
                    "name": tool.name, "description": tool.description or "", "parameters": tool.inputSchema}}
                for tool in self.tools]

    def close(self) -> None:
        """Close the session, stop the server process and the background loop."""
        if self.stop_event:
            self.loop.call_soon_threadsafe(self.stop_event.set)
            try:
                self.main_task.result(10)
            except Exception:
                pass
        self.loop.call_soon_threadsafe(self.loop.stop)
        self.log_file.close()


# ============================================================================ turn stats

class TurnStats:
    """Token usage and timing for one user turn, across every model call it makes.

    Attributes:
        started: ``perf_counter`` time the turn began (the user's message).
        model_seconds: Total time spent waiting on the model.
        prompt_tokens: Prompt tokens reported by the server, summed over calls.
        completion_tokens: Completion tokens reported by the server, summed over calls.
        calls: Number of model calls.
    """

    def __init__(self) -> None:
        """Start the turn clock."""
        self.started = time.perf_counter()
        self.model_seconds = 0.0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.calls = 0

    async def complete(self, **request: object) -> object:
        """Run a chat completion in a worker thread, recording its duration and token usage.

        Args:
            **request: Arguments for ``client.chat.completions.create``.

        Returns:
            The ``ChatCompletion`` response.
        """
        call_start = time.perf_counter()
        response = await asyncio.to_thread(client.chat.completions.create, **request)
        self.model_seconds += time.perf_counter() - call_start
        self.calls += 1
        if response.usage:
            self.prompt_tokens += response.usage.prompt_tokens or 0
            self.completion_tokens += response.usage.completion_tokens or 0
        return response

    def footer_html(self) -> str:
        """Render the stats line shown under the assistant's final answer.

        Returns:
            HTML for the footer, styled by ``STATS_CSS``.
        """
        elapsed = time.perf_counter() - self.started
        speed = self.completion_tokens / self.model_seconds if self.model_seconds else 0.0
        total = self.prompt_tokens + self.completion_tokens
        return (f'<div class="stats">'
                f'<span title="output tokens ÷ time waiting on the model (includes prompt processing)">'
                f'{speed:,.1f} tok/s</span>'
                f'<span title="{self.calls} model call(s)">{total:,} tokens '
                f'({self.prompt_tokens:,} in / {self.completion_tokens:,} out)</span>'
                f'<span title="time since your message, including tool calls">{elapsed:,.1f} s</span></div>')


# ============================================================================ context management

class Conversation:
    """Full message history for one chat, kept inside the model's context window.

    Compaction runs in stages until the estimate is under ``TARGET_AFTER`` of the usable window:
    shorten old tool results (full text archived, recallable via ``recall_tool_result``), fold old
    turns into a rolling summary, and as a last resort shorten current results and drop old turns.

    Attributes:
        messages: User, assistant (with tool calls) and tool messages in OpenAI format.
        summary: Rolling summary of compacted turns.
        archive: Full text of shortened tool results, keyed by tool call id.
        chars_per_token: Estimate ratio, recalibrated from the server's real token counts.
        last_prompt_tokens: Prompt size the server reported for the latest call.
        log: Human-readable record of each compaction.
    """

    def __init__(self) -> None:
        """Start an empty conversation."""
        self.messages: list[dict] = []
        self.summary = ""
        self.archive: dict[str, str] = {}
        self.chars_per_token = 3.2
        self.last_prompt_tokens = 0
        self.log: list[str] = []

    def system_messages(self) -> list[dict]:
        """Build the system prompt plus the rolling summary, if there is one.

        Returns:
            One or two system messages.
        """
        system = [{"role": "system", "content": SYSTEM_PROMPT}]
        if self.summary:
            system.append({"role": "system", "content": "Summary of earlier conversation:\n" + self.summary})
        return system

    def payload(self) -> list[dict]:
        """Build the full message list to send to the model.

        Returns:
            System messages followed by the conversation.
        """
        return self.system_messages() + self.messages

    def tokens(self, value: object) -> int:
        """Estimate the token count of a JSON-serialisable value.

        Args:
            value: Message, message list or tool list.

        Returns:
            Estimated number of tokens.
        """
        return int(len(to_json(value)) / self.chars_per_token)

    def estimate(self, tools: list[dict]) -> int:
        """Estimate the prompt size of the next request.

        Args:
            tools: Tool definitions sent with the request.

        Returns:
            Estimated prompt tokens.
        """
        return self.tokens(self.payload()) + self.tokens(tools)

    def breakdown(self, tools: list[dict]) -> dict[str, int]:
        """Split the estimated prompt size into the segments shown in the context bar.

        Args:
            tools: Tool definitions sent with the request.

        Returns:
            Estimated tokens per segment.
        """
        system = self.system_messages()
        return {
            "system + tools": self.tokens(system[:1]) + self.tokens(tools),
            "summary": self.tokens(system[1:]) if len(system) > 1 else 0,
            "chat": sum(self.tokens(message) for message in self.messages if message["role"] != "tool"),
            "tool results": sum(self.tokens(message) for message in self.messages if message["role"] == "tool"),
        }

    def calibrate(self, tools: list[dict], prompt_tokens: int, limit: int) -> None:
        """Update the chars-per-token ratio from the server's real prompt token count.

        Args:
            tools: Tool definitions sent with the request.
            prompt_tokens: ``usage.prompt_tokens`` reported by the server.
            limit: Context window of the model used.
        """
        self.last_prompt_tokens = prompt_tokens
        # Skip tiny samples and counts near the limit, where the server may have truncated the prompt.
        if 50 < prompt_tokens < 0.9 * limit:
            ratio = (len(to_json(self.payload())) + len(to_json(tools))) / prompt_tokens
            self.chars_per_token = 0.6 * self.chars_per_token + 0.4 * max(1.5, min(ratio, 6.0))

    def turn_starts(self) -> list[int]:
        """Find where each user turn starts.

        Returns:
            Indexes of the user messages in ``messages``.
        """
        return [index for index, message in enumerate(self.messages) if message["role"] == "user"]

    def shorten(self, message: dict) -> bool:
        """Replace a tool result with a short preview, archiving the full text.

        Args:
            message: A tool message, modified in place.

        Returns:
            True if the message was shortened, False if it was already short.
        """
        result_id, full_text = message["tool_call_id"], message["content"]
        if result_id in self.archive or len(full_text) <= TOOL_PREVIEW_CHARS:
            return False
        self.archive[result_id] = full_text
        message["content"] = (f"{full_text[:TOOL_PREVIEW_CHARS]}\n...[shortened, {len(full_text)} chars total. "
                              f"Full text: recall_tool_result(result_id='{result_id}')]")
        return True

    def recall(self, result_id: str, offset: int = 0, length: int = 1500) -> str:
        """Return part of an archived tool result (the local ``recall_tool_result`` tool).

        Args:
            result_id: Tool call id shown in the shortened result.
            offset: Character offset to start from.
            length: Number of characters to return.

        Returns:
            The requested slice, or an error listing the known ids.
        """
        full_text = self.archive.get(result_id)
        if full_text is None:
            return f"No archived result with id {result_id}. Known ids: {list(self.archive)}"
        return full_text[int(offset): int(offset) + int(length)]

    async def compact(self, model: str, tools: list[dict], force: bool = False,
                      stats: TurnStats | None = None, server_rejected: bool = False) -> dict | None:
        """Shrink the conversation if it is over the ``COMPACT_AT`` threshold.

        Args:
            model: Model in use (sets the window and writes the summary).
            tools: Tool definitions sent with the request.
            force: Compact even below the threshold.
            stats: Turn stats that summarisation calls are counted in, if any.
            server_rejected: The server said the prompt was too long; make the estimate more pessimistic.

        Returns:
            A report with ``entry`` (log line), ``before`` / ``after`` (estimated tokens),
            ``actions``, ``summary`` (the new rolling summary, or None if it didn't change) and
            ``archived`` (ids of tool results shortened this time), or None if nothing was needed.
        """
        if server_rejected:
            # The server disagreed with the estimate, so assume more tokens per character from now on.
            self.chars_per_token = max(1.5, self.chars_per_token * 0.8)
        budget = CONTEXT[model] - RESERVE_OUTPUT
        before = self.estimate(tools)
        if not force and before < COMPACT_AT * budget:
            return None
        archived_before = set(self.archive)
        new_summary: str | None = None
        target = TARGET_AFTER * budget
        actions: list[str] = []

        # Stage 1: shorten tool results from earlier turns; the current turn stays intact.
        starts = self.turn_starts()
        current_turn = starts[-1] if starts else len(self.messages)
        shortened = sum(self.shorten(message) for message in self.messages[:current_turn] if message["role"] == "tool")
        if shortened:
            actions.append(f"shortened {shortened} old tool result(s)")

        # Stage 2: fold everything before the last KEEP_TURNS turns into the rolling summary.
        starts = self.turn_starts()
        if self.estimate(tools) > target and len(starts) > KEEP_TURNS:
            cut = starts[-KEEP_TURNS]
            old_messages, self.messages = self.messages[:cut], self.messages[cut:]
            self.summary = new_summary = await self.summarise(model, old_messages, stats or TurnStats())
            actions.append(f"summarised {len(old_messages)} message(s)")

        # Stage 3 (last resort): shorten current results, then drop whole turns from the front.
        if self.estimate(tools) > budget:
            shortened = sum(self.shorten(message) for message in self.messages if message["role"] == "tool")
            if shortened:
                actions.append(f"shortened {shortened} current tool result(s)")
        while self.estimate(tools) > budget and len(self.turn_starts()) > 1:
            del self.messages[:self.turn_starts()[1]]
            actions.append("dropped oldest turn")

        if not actions and not force:
            return None
        after = self.estimate(tools)
        entry = f"{time.strftime('%H:%M:%S')}  {before:,} → {after:,} tok: " + (", ".join(actions) or "nothing to compact")
        self.log.append(entry)
        return {"entry": entry, "before": before, "after": after, "actions": actions, "summary": new_summary,
                "archived": [result_id for result_id in self.archive if result_id not in archived_before]}

    async def summarise(self, model: str, old_messages: list[dict], stats: TurnStats) -> str:
        """Merge old messages into the rolling summary, keeping facts from tool results.

        Args:
            model: Model that writes the summary.
            old_messages: Messages being removed from the conversation.
            stats: Turn stats the summarisation calls are counted in.

        Returns:
            The updated summary.
        """
        lines = []
        for message in old_messages:
            if message["role"] == "tool":
                lines.append("TOOL RESULT: " + self.archive.get(message["tool_call_id"], message["content"]))
            elif message.get("tool_calls"):
                calls = ", ".join(f"{call['function']['name']}({call['function']['arguments']})"
                                  for call in message["tool_calls"])
                lines.append(f"ASSISTANT CALLED: {calls}")
            else:
                lines.append(f"{message['role'].upper()}: {message['content']}")
        transcript = "\n".join(lines)
        # Summarise in chunks sized so each summarisation request itself fits in the window.
        chunk_chars = max(1000, int((CONTEXT[model] - RESERVE_OUTPUT - 600) * self.chars_per_token * 0.6))
        summary = self.summary
        for start in range(0, len(transcript), chunk_chars):
            prompt = (f"CURRENT SUMMARY:\n{summary or '(empty)'}\n\n"
                      f"NEW TRANSCRIPT:\n{transcript[start:start + chunk_chars]}")
            response = await stats.complete(
                model=model, temperature=0, max_tokens=400,
                messages=[{"role": "system", "content": SUMMARY_PROMPT}, {"role": "user", "content": prompt}])
            summary = (response.choices[0].message.content or summary).strip()
        return summary


# ============================================================================ agent loop

def is_context_error(error: Exception) -> bool:
    """Check whether a server error means the prompt was too long.

    Args:
        error: Error raised by the OpenAI client.

    Returns:
        True for context-length errors.
    """
    message = str(error).lower()
    return "context" in message and ("length" in message or "window" in message or "exceed" in message)


def limit_from_error(error: Exception) -> int | None:
    """Read the real context window from a server error, e.g. "maximum context length is 4096 tokens".

    Args:
        error: Error raised by the OpenAI client.

    Returns:
        The window size in tokens, or None if the message doesn't state it.
    """
    match = (re.search(r"maximum context length is (\d+)", str(error))
             or re.search(r"context (?:length|window)\D{0,20}(\d{3,7})", str(error)))
    return int(match.group(1)) if match else None


async def run_agent(conversation: Conversation, model: str, user_message: str) -> AsyncIterator[tuple[str, str | dict | None]]:
    """Answer a user message, calling MCP tools as needed and keeping the context in budget.

    Args:
        conversation: The chat's history, updated in place.
        model: Model to use.
        user_message: The new user message.

    Yields:
        ``(kind, data)`` events: ``context`` (sizes changed), ``compacted`` (log entry),
        ``tool_start`` / ``tool_end`` (tool name, arguments, result), ``final`` (answer text and
        ``TurnStats``) and ``error`` (message for the user).
    """
    stats = TurnStats()
    conversation.messages.append({"role": "user", "content": user_message})
    tools = bridge.openai_tools() + [RECALL_TOOL]

    async def request_completion(with_tools: bool = True) -> object:
        """Send the current conversation to the model, counting it in the turn stats."""
        tool_args = {"tools": tools} if with_tools else {}
        return await stats.complete(model=model, messages=conversation.payload(), temperature=0.2,
                                    max_tokens=RESERVE_OUTPUT, **tool_args)

    for _ in range(MAX_TOOL_STEPS):
        report = await conversation.compact(model, tools, stats=stats)
        if report:
            yield "compacted", report
        yield "context", None

        try:
            response = await request_completion()
        except BadRequestError as error:
            if not is_context_error(error):
                raise
            # The server knows the real window: correct ours, compact harder and retry once.
            real_limit = limit_from_error(error)
            if real_limit and real_limit < CONTEXT[model]:
                CONTEXT[model] = real_limit
            report = await conversation.compact(model, tools, force=True, stats=stats, server_rejected=True)
            report["actions"].append(f"server rejected the prompt; window is {CONTEXT[model]:,}")
            yield "compacted", report
            try:
                if conversation.estimate(tools) > CONTEXT[model] - RESERVE_OUTPUT:
                    raise error
                response = await request_completion()
            except BadRequestError as retry_error:
                if not is_context_error(retry_error):
                    raise
                del conversation.messages[conversation.turn_starts()[-1]:]
                yield "error", ("That message (plus its tool results) is too long for this model's context window. "
                                "Try a shorter message or a model with a bigger window.")
                return

        if response.usage:
            conversation.calibrate(tools, response.usage.prompt_tokens, CONTEXT[model])
        reply = response.choices[0].message

        if not reply.tool_calls:
            content = reply.content or ""
            if not content.strip():
                # Small models sometimes return nothing: retry once without tools.
                retry = await request_completion(with_tools=False)
                content = retry.choices[0].message.content or "(the model returned an empty reply)"
            conversation.messages.append({"role": "assistant", "content": content})
            yield "context", None
            yield "final", {"content": content, "stats": stats}
            return

        conversation.messages.append({"role": "assistant", "content": reply.content or "",
                                      "tool_calls": [call.model_dump() for call in reply.tool_calls]})
        for call in reply.tool_calls:
            try:
                arguments = json.loads(call.function.arguments or "{}")
                arguments = arguments if isinstance(arguments, dict) else {}
            except json.JSONDecodeError:
                arguments = {}
            yield "tool_start", {"name": call.function.name, "args": arguments}
            try:
                if call.function.name == "recall_tool_result":
                    result = conversation.recall(**arguments)
                else:
                    result = await bridge.acall(call.function.name, arguments)
            except Exception as error:
                result = f"Tool error: {error}"
            conversation.messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
            yield "tool_end", {"name": call.function.name, "result": result}

    conversation.messages.append({"role": "assistant", "content": "(stopped: too many tool calls)"})
    yield "final", {"content": "Stopped: too many tool calls in one turn.", "stats": stats}


# ============================================================================ UI

SEGMENT_COLORS = {"system + tools": "#64748b", "summary": "#a855f7", "chat": "#3b82f6", "tool results": "#f59e0b"}

CONTEXT_BAR_CSS = """
:host { width: 100%; }
.ctx { font: 13px/1.4 system-ui, sans-serif; padding: 10px 14px; border-radius: 12px;
       background: rgba(127,127,127,.08); border: 1px solid rgba(127,127,127,.2); }
.top { display: flex; gap: 10px; align-items: baseline; flex-wrap: wrap; margin-bottom: 8px; }
.top b { font-size: 14px; }
.pill { padding: 1px 8px; border-radius: 999px; background: rgba(99,102,241,.15); color: #6366f1; font-weight: 600; }
.num { margin-left: auto; font-variant-numeric: tabular-nums; opacity: .85; }
.track { position: relative; display: flex; height: 14px; border-radius: 7px; overflow: hidden;
         background: rgba(127,127,127,.18); }
.seg { height: 100%; transition: width .4s ease; }
.seg + .seg { box-shadow: inset 1px 0 rgba(255,255,255,.5); }
.reserve { position: absolute; right: 0; top: 0; bottom: 0;
           background: repeating-linear-gradient(45deg, rgba(127,127,127,.35) 0 4px, transparent 4px 8px); }
.trigger { position: absolute; top: -3px; bottom: -3px; width: 2px; background: #ef4444; opacity: .7; }
.legend { display: flex; gap: 14px; flex-wrap: wrap; margin-top: 7px; font-size: 12px; opacity: .85; }
.legend i { display: inline-block; width: 10px; height: 10px; border-radius: 3px; margin-right: 5px; vertical-align: -1px; }
.hatch { background: repeating-linear-gradient(45deg, rgba(127,127,127,.6) 0 2px, transparent 2px 4px); }
.ok .num b { color: #16a34a; } .warn .num b { color: #d97706; } .full .num b { color: #dc2626; }
"""

STATS_CSS = """
.stats { display: flex; gap: 6px; flex-wrap: wrap; font: 11.5px/1.6 system-ui, sans-serif; opacity: .75;
         font-variant-numeric: tabular-nums; }
.stats span { padding: 0 8px; border-radius: 999px; background: rgba(127,127,127,.12); }
"""

# Stretch steps to full width: shrink-to-fit cards mis-measure titles and wrap tool names mid-word.
STEP_OPTIONS = dict(
    user="Assistant", sizing_mode="stretch_width", layout_params={"sizing_mode": "stretch_width"},
    stylesheets=[":host { margin-block: 2px !important; } "
                 ".card-header { white-space: nowrap; align-items: center; padding: 2px 8px; min-height: 0; } "
                 ".card-title { font-size: 0.95em; }"],
)


def context_bar_html(conversation: Conversation, model: str, tools: list[dict]) -> str:
    """Render the context window bar.

    Args:
        conversation: The chat to measure.
        model: Selected model (sets the window size).
        tools: Tool definitions sent with each request.

    Returns:
        HTML for the bar, styled by ``CONTEXT_BAR_CSS``.
    """
    limit = CONTEXT[model]
    segments = conversation.breakdown(tools)
    used = sum(segments.values())
    budget = limit - RESERVE_OUTPUT
    state = "ok" if used < COMPACT_AT * budget else ("warn" if used < budget else "full")
    bar_segments = "".join(
        f'<div class="seg" style="width:{min(tokens / limit * 100, 100):.2f}%;background:{SEGMENT_COLORS[name]}" '
        f'title="{name}: {tokens:,} tokens"></div>'
        for name, tokens in segments.items() if tokens)
    legend = "".join(f'<span><i style="background:{SEGMENT_COLORS[name]}"></i>{name} {tokens:,}</span>'
                     for name, tokens in segments.items())
    server_count = f" · server: {conversation.last_prompt_tokens:,}" if conversation.last_prompt_tokens else ""
    return f"""<div class="ctx {state}">
      <div class="top"><b>Context window</b><span class="pill">{html.escape(model)}</span>
        <span class="num"><b>{used / limit * 100:.0f}%</b> · ~{used:,} / {limit:,} tokens{server_count}
        · compacted {len(conversation.log)}×</span></div>
      <div class="track">{bar_segments}
        <div class="reserve" style="width:{RESERVE_OUTPUT / limit * 100:.2f}%" title="reserved for the reply"></div>
        <div class="trigger" style="left:{COMPACT_AT * budget / limit * 100:.2f}%" title="auto-compact threshold"></div>
      </div>
      <div class="legend">{legend}<span><i class="hatch"></i>reply reserve {RESERVE_OUTPUT:,}</span>
        <span><i style="background:#ef4444;width:2px"></i>auto-compact</span></div></div>"""


def format_tool_result(result: str) -> str:
    """Format a tool result as a Markdown code block: indented JSON if it parses, plain text otherwise.

    Args:
        result: Raw tool output.

    Returns:
        Markdown code block, truncated to 1,500 characters.
    """
    try:
        return "```json\n" + json.dumps(json.loads(result), indent=2)[:1500] + "\n```"
    except ValueError:
        return "```text\n" + result[:1500] + "\n```"


def compaction_markdown(report: dict) -> str:
    """Describe a compaction for the chat: token change, actions, new summary and archived results.

    Args:
        report: Report returned by ``Conversation.compact``.

    Returns:
        Markdown text.
    """
    lines = [f"**{report['before']:,} → {report['after']:,} tokens** · {', '.join(report['actions']) or 'nothing to compact'}"]
    if report["summary"]:
        quoted_summary = "\n".join(f"> {line}" for line in report["summary"].splitlines())
        lines += ["", "**Rolling summary** (sent to the model in place of the older messages):", "", quoted_summary]
    if report["archived"]:
        ids = ", ".join(f"`{result_id}`" for result_id in report["archived"])
        lines += ["", f"**Archived tool results** (full text recallable with `recall_tool_result`): {ids}"]
    return "\n".join(lines)


def create_app() -> pn.template.FastListTemplate:
    """Build the chat app for one browser session, with its own conversation.

    Returns:
        The page template.
    """
    conversation = Conversation()
    tools = bridge.openai_tools() + [RECALL_TOOL]

    model_select = pn.widgets.Select(name="Model", options=MODELS, value=MODELS[0], sizing_mode="stretch_width")
    new_chat_button = pn.widgets.Button(name="New chat", icon="plus", button_type="primary", sizing_mode="stretch_width")
    compact_button = pn.widgets.Button(name="Compact now", icon="arrows-minimize", button_type="light",
                                       sizing_mode="stretch_width")
    context_bar = pn.pane.HTML(sizing_mode="stretch_width", stylesheets=[CONTEXT_BAR_CSS], margin=(5, 10))
    compaction_log = pn.pane.Markdown("_No compactions yet._", sizing_mode="stretch_width", styles={"font-size": "12px"})
    tool_list = pn.pane.Markdown("\n".join(f"- `{tool.name}`" for tool in bridge.tools)
                                 + "\n- `recall_tool_result` _(local)_", sizing_mode="stretch_width")

    def refresh(*_) -> None:
        """Redraw the context bar and the compaction log."""
        context_bar.object = context_bar_html(conversation, model_select.value, tools)
        compaction_log.object = "\n".join(f"- {entry}" for entry in conversation.log[-6:]) or "_No compactions yet._"

    model_select.param.watch(refresh, "value")

    async def respond(contents: str, user: str, instance: pn.chat.ChatInterface) -> AsyncIterator[dict]:
        """ChatInterface callback: run the agent and show tool calls and compactions as steps.

        Args:
            contents: The user's message.
            user: Name of the sender.
            instance: The chat interface.

        Yields:
            Assistant messages for the chat.
        """
        open_steps: dict[str, pn.chat.ChatStep] = {}
        async for kind, data in run_agent(conversation, model_select.value, contents):
            if kind == "context":
                refresh()
            elif kind == "compacted":
                # Open by default when a new summary was written, so it's visible without a click.
                title = "Context compacted: summary updated" if data["summary"] else "Context compacted"
                instance.add_step(compaction_markdown(data), title=title, status="success",
                                  collapsed=not data["summary"], **STEP_OPTIONS)
                refresh()
            elif kind == "tool_start":
                # A step's title can't change after creation; Panel switches to these titles with the status.
                title = f"Tool: {data['name']}"
                open_steps[data["name"]] = instance.add_step(
                    f"**args** `{to_json(data['args'])}`", title=title, success_title=title,
                    failed_title=f"{title} (failed)", status="running", **STEP_OPTIONS)
            elif kind == "tool_end":
                step = open_steps.pop(data["name"], None)
                if step:
                    step.stream("\n\n" + format_tool_result(data["result"]))
                    # ADAPT: works for FastMCP servers; if yours reports errors differently, use MCP's result.isError instead.
                    failed = data["result"].lstrip().lower().startswith(("error", "tool error"))
                    step.status = "failed" if failed else "success"
                    step.collapsed = True
                refresh()
            elif kind == "error":
                refresh()
                yield {"object": f"⚠️ {data}", "user": "Assistant"}
            else:
                yield {"object": data["content"], "user": "Assistant"}
                # Stats go in the footer of the final answer only (not on tool steps or errors).
                instance.objects[-1].footer_objects = [
                    pn.pane.HTML(data["stats"].footer_html(), stylesheets=[STATS_CSS], margin=0)]

    chat = pn.chat.ChatInterface(
        callback=respond, callback_user="Assistant", show_clear=False, show_rerun=False, show_undo=False,
        callback_exception="verbose", sizing_mode="stretch_both", min_height=500,
        message_params={"show_reaction_icons": False},
        # ADAPT: replace the ticket wording in the placeholder and the welcome message below.
        widgets=pn.chat.ChatAreaInput(placeholder="Ask about tickets..."))
    chat.send("Hi! I can look up support tickets through the MCP tool. Try **“what open tickets are there?”**",
              user="Assistant", respond=False)

    def start_new_chat(_) -> None:
        """Clear the chat and its history."""
        conversation.__init__()
        chat.clear()
        refresh()

    async def compact_now(_) -> None:
        """Force a compaction from the sidebar button and post its report in the chat."""
        report = await conversation.compact(model_select.value, tools, force=True)
        chat.send(pn.pane.Markdown(compaction_markdown(report), sizing_mode="stretch_width"),
                  user="Context", avatar="🗜️", respond=False)
        refresh()

    new_chat_button.on_click(start_new_chat)
    compact_button.on_click(compact_now)
    refresh()

    return pn.template.FastListTemplate(
        title="AI Harness", accent="#6366f1", theme_toggle=True, main_layout=None, sidebar_width=280,
        sidebar=[model_select, new_chat_button, compact_button,
                 pn.layout.Divider(), pn.pane.Markdown("#### MCP tools"), tool_list,
                 pn.layout.Divider(), pn.pane.Markdown("#### Compaction log"), compaction_log],
        main=[pn.Column(context_bar, chat, sizing_mode="stretch_both")],
    )


# ============================================================================ start

# ADAPT: change command/args if your MCP server isn't a Python script, e.g. command="node", args=["server.js"].
bridge = MCPBridge(StdioServerParameters(command=sys.executable, args=[MCP_SERVER_SCRIPT], env=MCP_ENV), MCP_LOG_PATH)
print("models:", CONTEXT)
# ADAPT: scratch out the whoami call: it only exists on this project's MCP server and crashes otherwise.
print("MCP tools:", [tool.name for tool in bridge.tools], "| whoami:", bridge.call("whoami", {}))

server = pn.serve(create_app, port=PANEL_PORT, show=False, threaded=True,
                  websocket_origin=PANEL_WS_ORIGINS, title="AI harness")


def shutdown() -> None:
    """Stop the Panel app and the MCP server (run in a new cell when finished)."""
    server.stop()
    bridge.close()
    print("stopped Panel + MCP server")


# JupyterHub sets JUPYTERHUB_SERVICE_PREFIX (e.g. /user/<name>/); plain Jupyter serves from "/".
app_url = f"{os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/').rstrip('/')}/proxy/{PANEL_PORT}/"
display(HTML(f'<a href="{app_url}" target="_blank" style="font-size:1.2em">Open AI harness in a new tab &rarr; {app_url}</a>'))